# rndresearch — Log Analysis

This notebook loads the structured JSONL logs produced by the rndresearch agent and provides
interactive analysis of:
- Run history and duration
- Papers found and summarised per topic
- Advanced filter acceptance rates
- LLM call durations and token usage

Logs are stored in `../logs/YYYY-MM-DD.jsonl` (one file per day).

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

LOG_DIR = Path("../logs")

# ── Load all JSONL log files ──────────────────────────────────────────────────
records = []
for log_file in sorted(LOG_DIR.glob("*.jsonl")):
    with open(log_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

df = pd.DataFrame(records)
if not df.empty:
    df["timestamp"] = pd.to_datetime(df["timestamp"])
    df = df.sort_values("timestamp").reset_index(drop=True)

print(f"Loaded {len(df)} log entries from {LOG_DIR}")
print(f"Event types: {df['event'].value_counts().to_dict() if not df.empty else 'none'}")

## 1. Run History

In [ ]:
runs = df[df["event"] == "run_start"].copy() if not df.empty else pd.DataFrame()
run_ends = df[df["event"] == "run_end"].copy() if not df.empty else pd.DataFrame()

if not runs.empty:
    run_summary = runs[["run_id", "timestamp", "days", "provider", "model",
                         "advanced_filter", "topics_count"]].copy()
    if not run_ends.empty:
        ends = run_ends[["run_id", "new_summaries", "skipped",
                          "filtered_out", "duration_ms"]]
        run_summary = run_summary.merge(ends, on="run_id", how="left")
        run_summary["duration_s"] = (run_summary["duration_ms"] / 1000).round(1)

    display(run_summary)
else:
    print("No run_start events found yet. Run the agent first.")

## 2. Papers Found Per Topic

In [ ]:
fetches = df[df["event"] == "arxiv_fetch"].copy() if not df.empty else pd.DataFrame()

if not fetches.empty:
    topic_stats = (
        fetches.groupby("topic")
        .agg(
            total_calls=("topic", "count"),
            total_in_window=("in_window", "sum"),
            avg_in_window=("in_window", "mean"),
            avg_duration_ms=("duration_ms", "mean"),
        )
        .round(1)
        .reset_index()
    )
    display(topic_stats)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(topic_stats["topic"], topic_stats["total_in_window"])
    ax.set_title("Total Papers Found Per Topic (all runs)")
    ax.set_ylabel("Papers in date window")
    ax.set_xlabel("Topic")
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No arxiv_fetch events found yet.")

## 3. Advanced Filter — Acceptance Rate Per Topic

In [ ]:
filters = df[df["event"] == "abstract_filter"].copy() if not df.empty else pd.DataFrame()

if not filters.empty:
    filters["accepted"] = filters["result"] == "accepted"
    filter_stats = (
        filters.groupby("topic")
        .agg(
            total=("result", "count"),
            accepted=("accepted", "sum"),
            rejected=("accepted", lambda x: (~x).sum()),
            acceptance_rate=("accepted", "mean"),
            avg_duration_ms=("duration_ms", "mean"),
        )
        .round(2)
        .reset_index()
    )
    display(filter_stats)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(filter_stats["topic"], filter_stats["acceptance_rate"] * 100)
    ax.set_title("Abstract Filter Acceptance Rate Per Topic")
    ax.set_ylabel("Acceptance rate (%)")
    ax.set_ylim(0, 100)
    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No abstract_filter events found. Run with --advanced-filter to see this data.")

## 4. LLM Summarisation — Durations and Token Usage

In [ ]:
summaries = df[df["event"] == "llm_summarize"].copy() if not df.empty else pd.DataFrame()

if not summaries.empty:
    print(f"Total summaries generated: {len(summaries)}")
    print(f"Providers used: {summaries['provider'].value_counts().to_dict()}")
    print(f"Models used: {summaries['model'].value_counts().to_dict()}")

    stats = summaries[["provider", "model", "duration_ms",
                         "tokens_in", "tokens_out"]].describe().round(1)
    display(stats)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Duration histogram
    axes[0].hist(summaries["duration_ms"].dropna() / 1000, bins=20, edgecolor="black")
    axes[0].set_title("LLM Summarisation Duration")
    axes[0].set_xlabel("Duration (seconds)")
    axes[0].set_ylabel("Count")

    # Token usage scatter
    valid_tokens = summaries.dropna(subset=["tokens_in", "tokens_out"])
    if not valid_tokens.empty:
        axes[1].scatter(valid_tokens["tokens_in"], valid_tokens["tokens_out"], alpha=0.6)
        axes[1].set_title("Token Usage (input vs output)")
        axes[1].set_xlabel("Input tokens")
        axes[1].set_ylabel("Output tokens")
    else:
        axes[1].text(0.5, 0.5, "No token data available",
                     ha="center", va="center", transform=axes[1].transAxes)

    plt.tight_layout()
    plt.show()
else:
    print("No llm_summarize events found yet.")

## 5. Timeline — Events Over Time

In [ ]:
if not df.empty:
    event_counts = (
        df.set_index("timestamp")
        .groupby([pd.Grouper(freq="D"), "event"])
        .size()
        .unstack(fill_value=0)
    )
    ax = event_counts.plot(kind="bar", stacked=True, figsize=(12, 5))
    ax.set_title("Agent Events Per Day")
    ax.set_xlabel("Date")
    ax.set_ylabel("Event count")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()
else:
    print("No log data available yet.")

## 6. Raw Log Explorer

In [ ]:
# Filter by event type or run_id to inspect specific entries
# Examples:
#   df[df["event"] == "llm_summarize"][["timestamp", "paper_id", "title", "duration_ms"]]
#   df[df["run_id"] == "20260228T112500Z"]

if not df.empty:
    display(df[["timestamp", "run_id", "event"]].tail(20))
else:
    print("No log data available yet. Run the agent to generate logs.")